<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/Biomass_using_XGBOOST_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rioxarray

In [ ]:
!pip install --upgrade xee

In [ ]:
!pip install -U geemap

In [ ]:
import rioxarray
import ee
import geemap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
import xee
import geopandas as gpd

In [ ]:
ee.Authenticate()
ee.Initialize(project = "ee-grmntfrancis0",
              opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
import geemap

In [ ]:
map = geemap.Map()
map

In [ ]:
point = map.draw_last_feature.geometry()
point

In [ ]:
border = (
    ee.FeatureCollection("FAO/GAUL/2015/level1")
    .filterBounds(point)
)

In [ ]:
map.addLayer(border)

In [ ]:
vec = geemap.ee_to_gdf(border)
vec

In [ ]:
biomass = (
    ee.ImageCollection("NASA/ORNL/biomass_carbon_density/v1").select('agb')
    .first().rename('biomass')
)
biomass

In [ ]:
ndvi = (
    ee.ImageCollection("MODIS/061/MOD13A2")
    .select('NDVI','EVI')
    .filterDate('2010','2011')
    .mean().rename('ndvi', 'evi')
)
ndvi

In [ ]:
lai = (
    ee.ImageCollection("MODIS/061/MOD15A2H")
    .select('Lai_500m', 'Fpar_500m')
    .filterDate('2010', '2011')
    .mean().rename('lai', 'fpar')
)
lai

In [ ]:
temp = (
    ee.ImageCollection("MODIS/061/MYD11A2")
    .select('LST_Day_1km')
    .filterDate('2010', '2011')
    .mean().rename('temp')
)
temp

In [ ]:
pr = (
    ee.ImageCollection("NASA/GPM_L3/IMERG_MONTHLY_V07")
    .select('precipitation')
    .filterDate('2010','2011')
    .sum().rename('pr')
)
pr

In [ ]:
lc = (
    ee.ImageCollection("MODIS/061/MCD12Q1")
    .select('LC_Type1')
    .filterDate('2010','2011')
    .first().rename('lc')
)
lc

In [ ]:
stack = ee.Image.cat([
    ndvi, lai, temp, pr, lc, biomass
])

stack

In [ ]:
import xarray as xr

In [ ]:
from jinja2.nodes import ExprStmt
ds = xr.open_dataset(
    stack,
    engine = 'ee',
    crs = 'EPSG:4326',
    scale = 0.01,
    geometry = border.geometry()
)
ds

In [ ]:
df = ds.to_dataframe().dropna()
df

In [ ]:
x = df[['ndvi', 'lai', 'temp', 'pr', 'lc', 'fpar', 'evi']]
y = df['biomass']

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)


In [ ]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators = 100,
    random_state = 42
)

model.fit(x_train, y_train)

In [ ]:
y_pred = model.predict(x_test)

from sklearn.metrics import mean_squared_error, r2_score


In [ ]:
import numpy as np

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f'RMSE:{rmse}')
r2 = r2_score(y_test, y_pred)
print(f'R2:{r2}')

In [ ]:
df_clipped = ds_clip.to_dataframe().dropna()
df_clipped['predicted_biomass'] = model.predict(df_clipped[['ndvi', 'lai', 'temp', 'pr', 'lc', 'fpar', 'evi']])

df_clipped

In [ ]:
# 1. Convert and drop NaNs from specific features
df_clipped = ds_clip.to_dataframe().dropna(subset=['ndvi', 'lai', 'temp', 'pr', 'lc', 'fpar', 'evi']).copy()

# 2. Predict and assign
features = ['ndvi', 'lai', 'temp', 'pr', 'lc', 'fpar', 'evi']
df_clipped['predicted_biomass'] = model.predict(df_clipped[features])

In [ ]:
xarr_clipped = df_clipped.to_xarray().sortby('x').sortby('y')
xarr_clipped

In [ ]:
fig, ax = plt.subplots(1,2 , figsize = (10,5))
plt.tight_layout()

xarr.biomass.plot(
    ax = ax[0],
    x = 'lon',
    y = 'lat',
    robust = True
)


xarr.predicted_biomass.plot(
    ax = ax[1],
    x = 'lon',
    y = 'lat',
    robust = True
)


In [ ]:
import geopandas as gpd

In [ ]:
vec.plot()

In [ ]:
vec.crs

In [ ]:
vec.geometry

In [ ]:
ds_rename = ds.rename({"lon": "x", "lat": "y"})

In [ ]:
ds_clip = ds_rename.rio.clip(vec.geometry, vec.crs)


In [ ]:
import pandas as pd

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (10,5))
plt.tight_layout()

xarr_clipped.biomass.plot(
    ax = ax[0],
    x = 'x',
    y = 'y',
    robust = True,
    cbar_kwargs = {'orientation': 'vertical', 'label': 'biomass Mg/ha', 'shrink': 0.5}
)
ax[0].set_aspect("equal")
ax[0].set_title('biomass 2010-2011')

xarr_clipped.predicted_biomass.plot(
    ax = ax[1],
    x = 'x',
    y = 'y',
    robust = True,
    cbar_kwargs = {'orientation': 'vertical', 'label': 'biomass Mg/ha', 'shrink': 0.5}
)
ax[1].set_aspect("equal")
ax[1].set_title('predicted biomass 2010-2011')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate the mean biomass and predicted biomass along the 'y' (latitude) axis
mean_biomass_along_x = xarr_clipped.biomass.mean(dim='y')
mean_predicted_biomass_along_x = xarr_clipped.predicted_biomass.mean(dim='y')

# Plotting trends along the 'x' (longitude) axis
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
mean_biomass_along_x.plot(label='Actual biomass', color='blue')
mean_predicted_biomass_along_x.plot(label='Predicted biomass', color='orange', linestyle='--')
plt.title('Average biomass 2010-2011')
plt.xlabel('Longitude (x)')
plt.ylabel('Average biomass Mg/ha')
plt.legend()
plt.grid(True)

# Calculate the mean biomass and predicted biomass along the 'x' (longitude) axis
mean_biomass_along_y = xarr_clipped.biomass.mean(dim='x')
mean_predicted_biomass_along_y = xarr_clipped.predicted_biomass.mean(dim='x')

# Plotting trends along the 'y' (latitude) axis
plt.subplot(1, 2, 2)
mean_biomass_along_y.plot(label='Actual biomass', color='blue')
mean_predicted_biomass_along_y.plot(label='Predicted biomass', color='orange', linestyle='--')
plt.title('Average biomass 2010-2011')
plt.xlabel('Latitude (y)')
plt.ylabel('Average biomass Mg/ha')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('average_biomass_line_chart.png') # Export the line chart
plt.show()